In [181]:
import pandas as pd
import numpy as np
import yfinance as yf
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from backtesting import Strategy, Backtest
from datetime import datetime, timedelta
from scipy.stats import linregress
import warnings
warnings.filterwarnings("ignore")
import plotly.io as pio
pio.renderers.default = "notebook_connected"

In [182]:
df = yf.download("QQQ", period="10y", interval="1d")
df.columns = df.columns.get_level_values(0)
df = df.reset_index()
df['Date'] = pd.to_datetime(df['Date'])
df

[*********************100%***********************]  1 of 1 completed


Price,Date,Close,High,Low,Open,Volume
0,2016-05-16,99.522102,99.885457,98.348183,98.441357,18311900
1,2016-05-17,98.273659,99.727078,98.012782,99.438259,29932700
2,2016-05-18,98.627701,99.158758,97.863723,98.050056,27524400
3,2016-05-19,98.115295,98.487969,97.397904,98.245730,27334600
4,2016-05-20,99.196053,99.596676,98.450705,98.506611,29549600
...,...,...,...,...,...,...
2509,2026-05-08,711.229980,711.229980,699.500000,699.919983,44320400
2510,2026-05-11,713.289978,714.590027,708.909973,710.359985,36019100
2511,2026-05-12,707.239990,710.179993,696.640015,708.219971,45873000
2512,2026-05-13,714.710022,716.650024,704.830017,709.960022,39709700


In [183]:
# df['Vol'] = df['Volume'] / df['Close']

df['Vol'] = df['Volume'] * df['Close']

# relVol = df['Volume'] / df['Close']
# df['Vol'] = relVol / relVol.rolling(20).mean()

def Williams(data, n):
    HiHi = df['Vol'].rolling(n).max()
    LoLo = df['Vol'].rolling(n).min()
    Close = df['Vol']
    will = ((HiHi - Close) / (HiHi - LoLo)) * 100
    will1 = will.rolling(3).mean()
    return will1

df['williams'] = Williams(df, 17)
df.dropna(inplace=True)
df

Price,Date,Close,High,Low,Open,Volume,Vol,williams
18,2016-06-10,101.497269,101.925841,101.143226,101.730189,32965000,3.345857e+09,63.044940
19,2016-06-13,100.649429,101.487943,100.518995,100.919617,25154700,2.531806e+09,49.852351
20,2016-06-14,100.649429,101.012784,99.913401,100.379241,24588500,2.474818e+09,41.290293
21,2016-06-15,100.360626,101.031437,100.192923,100.891684,24605700,2.469443e+09,52.475380
22,2016-06-16,100.658768,100.751934,99.223979,99.866841,32079900,3.229123e+09,43.575989
...,...,...,...,...,...,...,...,...
2509,2026-05-08,711.229980,711.229980,699.500000,699.919983,44320400,3.152200e+10,36.915049
2510,2026-05-11,713.289978,714.590027,708.909973,710.359985,36019100,2.569206e+10,40.046827
2511,2026-05-12,707.239990,710.179993,696.640015,708.219971,45873000,3.244322e+10,29.640999
2512,2026-05-13,714.710022,716.650024,704.830017,709.960022,39709700,2.838092e+10,33.724616


In [184]:
def signal(data):
    signal = [0] * len(df)
    for i in range(2,len(df)):
        if (data.williams.iloc[i-1] < 50) and (data.williams.iloc[i] > 50):
            signal[i] = 1
        elif (data.williams.iloc[i-1] > 90) and (data.williams.iloc[i] < 90):
            signal[i] = 2
        else:
            signal[i] = 0
        df["signal"] = signal
        
signal(df)


In [185]:
def long_entries(x):
    offset = 0.002
    if x['signal']==1:
        return x['Low'] * (1-offset)
    else:
        return np.nan

df['long_entries'] = df.apply(lambda x: long_entries(x), axis=1)

def short_entries(x):
    offset = 0.002
    if x['signal']==2:
        return x['High'] * (1+offset)
    else:
        return np.nan

df['short_entries'] = df.apply(lambda x: short_entries(x), axis=1)


print(df['signal'].value_counts())
df.shape


signal
0    2227
1     182
2      87
Name: count, dtype: int64


(2496, 11)

In [186]:
df.set_index('Date', inplace=True)

In [187]:
bar = 2251
df1 = df[bar:bar+250].copy()

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, 
                    row_heights=[0.68,0.32], 
                    vertical_spacing=0.05)

fig.add_trace(go.Candlestick(x = df1.index, 
                            open = df1['Open'],
                            high = df1['High'],
                            low = df1['Low'],
                            close = df1['Close'],
                            increasing_line_color = 'rgba(19,156,19,0.8)',
                            decreasing_line_color = 'rgba(175,07,49,0.8)',
                            name = 'QQQ'),
                            row=1, col=1)

fig.add_scatter(x=df1.index, y=df1['long_entries'], mode="markers",
                marker=dict(size=7, symbol='arrow-up', color="White"),
                name="Long Entries")

fig.add_scatter(x=df1.index, y=df1['short_entries'], mode="markers",
                marker=dict(size=7, symbol='cross', color="gold"),
                name="Short Entries")

fig.add_trace(go.Scatter(x=df1.index, 
                         y=df1.williams, 
                         line=dict(color='red', width=2),
                         name='%R'),
                         row=2, col=1)

fig.add_hline(y=80, 
              line_width=0.5, 
              line_color="grey", 
              row=2, col=1)

fig.add_hline(y=50, 
              line_width=0.5, 
              line_color="grey", 
              row=2, col=1)

fig.add_hline(y=20, 
              line_width=0.5, 
              line_color="grey", 
              row=2, col=1)

fig.update_layout(autosize=False, width=1100, height=700, 
                  xaxis_rangeslider_visible=False, 
                  template="plotly_dark")

fig.update_yaxes(gridcolor="#171717") 
fig.update_xaxes(gridcolor="#171717")

fig.update_xaxes(
    rangebreaks=[
        dict(bounds=["sat", "mon"]),
    ])

fig.show()

In [188]:
def SIGNAL():
    return df.signal

class MyStrat(Strategy):
    
    def init(self):
        super().init()
        self.signal = self.I(SIGNAL)
    
    def next(self):
        super().next()
        
        price = self.data.Close[-1]
        
        if self.signal==1: 
            if self.position.is_short or not self.position:
                self.position.close()
                self.buy(size=0.99, tp=1.09*price)
        
        elif self.signal==2:
            if self.position.is_long or not self.position:
                self.position.close()
                self.sell(size=0.99, tp=0.98*price, sl=1.02*price)
                         
bt = Backtest(df, MyStrat, cash=100_000, margin=1, exclusive_orders=True, commission=0.0005)
stats = bt.run()
stats

Start                     2016-06-10 00:00:00
End                       2026-05-14 00:00:00
Duration                   3625 days 00:00:00
Exposure Time [%]                    79.04647
Equity Final [$]                  447598.3203
Equity Peak [$]                  465292.33563
Commissions [$]                   33757.25227
Return [%]                          347.59832
Buy & Hold Return [%]               609.83193
Return (Ann.) [%]                    16.33615
Volatility (Ann.) [%]                21.25464
CAGR [%]                             10.98083
Sharpe Ratio                          0.76859
Sortino Ratio                         1.28793
Calmar Ratio                          0.58151
Alpha [%]                             7.57853
Beta                                  0.55756
Max. Drawdown [%]                   -28.09284
Avg. Drawdown [%]                    -2.77055
Max. Drawdown Duration      559 days 00:00:00
Avg. Drawdown Duration       31 days 00:00:00
# Trades                          

In [189]:
trades = stats['_trades']
trades['CumulativePnL'] = trades['PnL'].cumsum()

fig_trades = go.Figure()

fig_trades.add_trace(go.Scatter(x=trades['EntryTime'], 
                                      y=trades['CumulativePnL'], 
                                      mode='lines', 
                                      name='Cumulative PnL', 
                                      line=dict(color='#00df9a')))

fig_trades.update_layout(title='Vol Strategy PnL',
                         template="plotly_dark",
                         autosize=False,
                         width=1100,
                         height=700,
                        )

fig_trades.update_yaxes(gridcolor="#171717")
fig_trades.update_xaxes(gridcolor="#171717")

fig_trades.show()

In [190]:
def randomised_trades(trades):
    cumulative_return = [0]

    for pct in trades['ReturnPct']:
        cumulative_return.append(cumulative_return[-1] + (pct * 100))

    return cumulative_return

simulations = 100
curves = []

for i in range(simulations):
    new_trades = trades.sample(frac=1).reset_index(drop=True)
    equity_curve = randomised_trades(new_trades)
    curves.append(equity_curve)

mc = go.Figure()

for equity_curve in curves:
    mc.add_trace(go.Scatter(y=equity_curve, mode='lines', opacity=0.6, showlegend=False))

mc.update_layout(
    title='Vol Monte Carlo Simulation',
    xaxis_title='Trade Number',
    yaxis_title='Equity',
    template="plotly_dark",
    autosize=False,
    width=1100,
    height=700,
)

mc.update_yaxes(gridcolor="#171717")
mc.update_xaxes(gridcolor="#171717")

mc.show()